# 08 - Fine-tuned RAG Generation Evaluation

Runs RAG with the same base LLM checkpoint plus a LoRA adapter. Retrieval corpus, retriever mode, top-k context, prompt, and decoding settings must match Base RAG.

In [ ]:
from pathlib import Path
import json
import sys
import importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path = [str(DRIVE_ROOT)] + [p for p in sys.path if p != str(DRIVE_ROOT)]
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
index_root = DRIVE_ROOT / config.get('official_index_root', 'indexes/official_law_v3')
output_dir = DRIVE_ROOT / 'outputs/generation_eval'
llm_model = config.get('base_llm_model', 'google/gemma-2-2b-it')

# Use smoke adapter first. Change to gemma_2_2b_it_lora_combined_v1 after full/medium training.
adapter_path = DRIVE_ROOT / 'models/adapters/gemma_2_2b_it_lora_smoke'

for path in [benchmark_csv, index_root / 'index_manifest.json', adapter_path]:
    if not path.exists():
        raise FileNotFoundError(path)

llm_model, adapter_path

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = {
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'bitsandbytes': 'bitsandbytes',
    'peft': 'peft',
    'sentence_transformers': 'sentence-transformers',
    'faiss': 'faiss-cpu',
    'rank_bm25': 'rank-bm25',
}

missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All fine-tuned RAG dependencies are already installed.')

In [ ]:
import torch
from src.generation import run_rag_generation

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Smoke eval first. Set limit=None and change filenames after confirming output quality.
limit = 10

run_config = run_rag_generation(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=output_dir / 'finetuned_rag_predictions_v1_smoke.csv',
    output_run_config_json=output_dir / 'finetuned_rag_run_config_v1_smoke.json',
    llm_model=llm_model,
    adapter_path=adapter_path,
    system_name='finetuned_rag_smoke',
    retriever_mode='dense',
    top_k_context=10,
    candidate_k=30,
    dense_weight=1.0,
    bm25_weight=0.0,
    device=device,
    max_new_tokens=384,
    temperature=0.2,
    top_p=0.9,
    limit=limit,
    load_in_4bit=True,
)

run_config

In [ ]:
from src.evaluation_qa import evaluate_generation_predictions

summary = evaluate_generation_predictions(
    predictions_csv=output_dir / 'finetuned_rag_predictions_v1_smoke.csv',
    output_eval_csv=output_dir / 'finetuned_rag_eval_v1_smoke.csv',
    output_summary_json=output_dir / 'finetuned_rag_eval_summary_v1_smoke.json',
)

summary

In [ ]:
import pandas as pd

base = pd.read_csv(output_dir / 'base_rag_eval_v1.csv', dtype=str, keep_default_na=False)
ft = pd.read_csv(output_dir / 'finetuned_rag_eval_v1_smoke.csv', dtype=str, keep_default_na=False)
preview_cols = ['question_id', 'question', 'generated_answer', 'retrieved_citations', 'citation_present', 'citation_gold_match']
ft[preview_cols].head(5)

After the smoke adapter output looks sane, train the medium adapter in notebook `07`, then run the final Fine-tuned RAG evaluation with the medium adapter and filenames without `_smoke`:

- Adapter: `models/adapters/gemma_2_2b_it_lora_combined_v1`
- `finetuned_rag_predictions_v1.csv`
- `finetuned_rag_run_config_v1.json`
- `finetuned_rag_eval_v1.csv`
- `finetuned_rag_eval_summary_v1.json`

Do not change retrieval settings from Base RAG.

In [ ]:
# Final Fine-tuned RAG run with the medium adapter from notebook 07.
# Run this after `models/adapters/gemma_2_2b_it_lora_combined_v1` exists.
medium_adapter_path = DRIVE_ROOT / 'models/adapters/gemma_2_2b_it_lora_combined_v1'
if not medium_adapter_path.exists():
    raise FileNotFoundError(f'Medium adapter not found yet: {medium_adapter_path}')

final_run_config = run_rag_generation(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=output_dir / 'finetuned_rag_predictions_v1.csv',
    output_run_config_json=output_dir / 'finetuned_rag_run_config_v1.json',
    llm_model=llm_model,
    adapter_path=medium_adapter_path,
    system_name='finetuned_rag',
    retriever_mode='dense',
    top_k_context=10,
    candidate_k=30,
    dense_weight=1.0,
    bm25_weight=0.0,
    device=device,
    max_new_tokens=384,
    temperature=0.2,
    top_p=0.9,
    limit=None,
    load_in_4bit=True,
)

final_summary = evaluate_generation_predictions(
    predictions_csv=output_dir / 'finetuned_rag_predictions_v1.csv',
    output_eval_csv=output_dir / 'finetuned_rag_eval_v1.csv',
    output_summary_json=output_dir / 'finetuned_rag_eval_summary_v1.json',
)

final_summary